In [112]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np

In [ ]:
torch.cuda.is_available()
df = pd.read_csv("cleaned_final.csv")
#df = pd.read_csv("RTN_test.csv")

features = ['x', 'y', 'z', 'vx', 'vy', 'vz', 'days_offset']
targets = ['x_error', 'y_error', 'z_error']

#features = ['x', 'y', 'z', 'vx', 'vy', 'vz', 'days_offset']
#targets = ['r_error', 't_error', 'n_error']

X = df[features].values
y = df[targets].values

#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
split_idx = int(len(df) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

df_test = df.iloc[split_idx:].reset_index(drop=True)

scaler_X = StandardScaler()
X_train = scaler_X.fit_transform(X_train)
X_test = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train = scaler_y.fit_transform(y_train)
y_test = scaler_y.transform(y_test)

# X_tensor = torch.tensor(X, dtype=torch.float32)
# y_tensor = torch.tensor(y, dtype=torch.float32)
# dataset = TensorDataset(X_tensor, y_tensor)

train_dataset = TensorDataset(  
    torch.tensor(X_train, dtype=torch.float32), 
    torch.tensor(y_train, dtype=torch.float32)
)
test_dataset = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32), 
    torch.tensor(y_test, dtype=torch.float32)
)

train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)


train: 7562 wierszy, test: 548 wierszy
train do: 2025-06-20 10:37:31.923840 | test od: 2025-07-01 12:43:44.102784


In [114]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(7, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128,128),
            nn.ReLU(),
            nn.Linear(128, 3)
        )

    def forward(self, x):
        return self.network(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Model().to(device=device)

loss_fn = nn.L1Loss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-2)

epochs = 1000
for epoch in range(epochs):
    model.train()
    ov_loss = 0.0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        y_pred = model(batch_X)
        loss = loss_fn(y_pred, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        ov_loss += loss.item() * batch_X.size(0)

    epoch_loss = ov_loss / len(train_dataset)

    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss:.4f}")

Epoch [100/1000], Loss: 0.2187
Epoch [200/1000], Loss: 0.1522
Epoch [300/1000], Loss: 0.1194
Epoch [400/1000], Loss: 0.0925
Epoch [500/1000], Loss: 0.1083
Epoch [600/1000], Loss: 0.0925
Epoch [700/1000], Loss: 0.0837
Epoch [800/1000], Loss: 0.0823
Epoch [900/1000], Loss: 0.0841
Epoch [1000/1000], Loss: 0.1018


In [115]:
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        y_pred = model(batch_X)
        all_preds.append(y_pred.cpu().numpy())
        all_targets.append(batch_y.cpu().numpy())

preds_array = np.vstack(all_preds)
targets_array = np.vstack(all_targets)

preds_km = scaler_y.inverse_transform(preds_array)
targets_km = scaler_y.inverse_transform(targets_array)

err3d_model = np.linalg.norm(preds_km - targets_km, axis=1)
err3d_raw_sgp4 = np.linalg.norm(targets_km, axis=1)

df_test['Err3D_Model'] = err3d_model
df_test['Err3D_SGP4'] = err3d_raw_sgp4

print("\n=== PODSUMOWANIE OGÓLNE 3D [km] ===")
print(f"SGP4  -> Średnia: {err3d_raw_sgp4.mean():.3f} | Mediana: {np.median(err3d_raw_sgp4):.3f} | 90th perc (ogon): {np.percentile(err3d_raw_sgp4, 90):.3f}")
print(f"Model -> Średnia: {err3d_model.mean():.3f} | Mediana: {np.median(err3d_model):.3f} | 90th perc (ogon): {np.percentile(err3d_model, 90):.3f}")

print("\n=== BŁĄD 3D W ROZBICIU NA DAYS_OFFSET [km] ===")
summary = df_test.groupby('days_offset').apply(lambda g: pd.Series({
    'Liczba_Próbek': len(g),
    'Mediana_SGP4': g['Err3D_SGP4'].median(),
    'Mediana_Model': g['Err3D_Model'].median(),
    'Średnia_SGP4': g['Err3D_SGP4'].mean(),
    'Średnia_Model': g['Err3D_Model'].mean(),
    '90perc_SGP4': np.percentile(g['Err3D_SGP4'], 90),
    '90perc_Model': np.percentile(g['Err3D_Model'], 90)
}))

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print(summary.round(3).to_string())


=== PODSUMOWANIE OGÓLNE 3D [km] ===
SGP4  -> Średnia: 3.449 | Mediana: 2.230 | 90th perc (ogon): 8.302
Model -> Średnia: 2.794 | Mediana: 1.942 | 90th perc (ogon): 5.752

=== BŁĄD 3D W ROZBICIU NA DAYS_OFFSET [km] ===
             Liczba_Próbek  Mediana_SGP4  Mediana_Model  Średnia_SGP4  Średnia_Model  90perc_SGP4  90perc_Model
days_offset                                                                                                    
0                     50.0         0.934          1.447         1.267          1.679        2.687         2.817
1                     50.0         0.999          1.396         1.317          1.746        2.664         3.150
2                     50.0         1.197          1.262         1.376          1.688        2.546         3.025
3                     50.0         1.316          1.245         1.459          1.607        2.412         2.981
4                     50.0         1.593          1.304         1.728          1.655        2.796         3.2

C:\Users\kozak\AppData\Local\Temp\ipykernel_1808\2958093815.py:29: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  summary = df_test.groupby('days_offset').apply(lambda g: pd.Series({


In [116]:
# from sklearn.ensemble import GradientBoostingRegressor
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import mean_absolute_error

# # Szybki test na obecnych danych
# X_train, X_test, y_train, y_test = train_test_split(X, y[:, 0], test_size=0.2, shuffle=False) # Chronologicznie!

# gbr = GradientBoostingRegressor(n_estimators=100, random_state=42)
# gbr.fit(X_train, y_train)
# preds = gbr.predict(X_test)

# print("MAE Gradient Boosting dla osi X:", mean_absolute_error(y_test, preds))

In [117]:
N = 10  # ile ostatnich wierszy pokazać

view = pd.DataFrame({
    'target_epoch':  df_test['target_epoch'].values,
    'days_offset':   df_test['days_offset'].values,

    # prawdziwy błąd SGP4 (prawda_SP3 - SGP4), bez korekty
    'SGP4_dx': targets_km[:, 0],
    'SGP4_dy': targets_km[:, 1],
    'SGP4_dz': targets_km[:, 2],

    # korekta przewidziana przez model
    'Model_dx': preds_km[:, 0],
    'Model_dy': preds_km[:, 1],
    'Model_dz': preds_km[:, 2],

    # odległość od prawdy w 3D: bez korekty vs z korektą
    'Dist_SGP4 [km]':  np.linalg.norm(targets_km, axis=1),
    'Dist_Model [km]': np.linalg.norm(targets_km - preds_km, axis=1),
})
view['Poprawa [km]'] = view['Dist_SGP4 [km]'] - view['Dist_Model [km]']  # >0 = model pomógł

pd.set_option('display.width', 1000)
pd.set_option('display.max_columns', None)
print(view.tail(N).round(3).to_string(index=False))

       target_epoch  days_offset  SGP4_dx  SGP4_dy  SGP4_dz  Model_dx  Model_dy  Model_dz  Dist_SGP4 [km]  Dist_Model [km]  Poprawa [km]
2025-08-31 12:00:00            0   -1.049   -0.601   -0.603    -0.888    -0.960    -1.182           1.351            0.700         0.651
2025-09-01 12:00:00            1   -1.522   -0.911   -0.564    -1.187    -1.227    -1.255           1.861            0.830         1.032
2025-09-02 12:00:00            2   -1.672   -0.973   -0.330    -1.155    -1.664    -1.022           1.962            1.106         0.856
2025-09-03 12:00:00            3   -1.400   -0.690    0.071    -1.260    -1.621    -0.841           1.562            1.310         0.252
2025-09-04 12:00:00            4   -0.681   -0.025    0.539    -1.569    -1.621    -0.521           0.868            2.112        -1.243
2025-09-05 12:00:00            5    0.423    0.973    0.929    -1.599    -1.471    -0.063           1.410            3.323        -1.913
2025-09-06 12:00:00            6    1.780

In [118]:
view = pd.DataFrame({
    'tle_epoch':   df_test['tle_epoch'].values,
    'target_epoch': df_test['target_epoch'].values,
    'days_offset': df_test['days_offset'].values,

    # surowy wynik SGP4 (pozycja, km)
    'SGP4_x': df_test['x'].values,
    'SGP4_y': df_test['y'].values,
    'SGP4_z': df_test['z'].values,

    # surowy wynik modelu (przewidziany błąd, km)
    'Model_x': preds_km[:, 0],
    'Model_y': preds_km[:, 1],
    'Model_z': preds_km[:, 2],
})
print(view.tail(10).round(3).to_string(index=False))

              tle_epoch        target_epoch  days_offset    SGP4_x    SGP4_y     SGP4_z  Model_x  Model_y  Model_z
2025-08-31 08:02:31.884 2025-08-31 12:00:00            0 -5709.110 28421.404 -30491.106   -0.888   -0.960   -1.182
2025-08-31 08:02:31.884 2025-09-01 12:00:00            1 -5700.382 28076.005 -30812.216   -1.187   -1.227   -1.255
2025-08-31 08:02:31.884 2025-09-02 12:00:00            2 -5703.653 27730.510 -31124.120   -1.155   -1.664   -1.022
2025-08-31 08:02:31.884 2025-09-03 12:00:00            3 -5718.982 27385.023 -31426.981   -1.260   -1.621   -0.841
2025-08-31 08:02:31.884 2025-09-04 12:00:00            4 -5746.347 27039.730 -31720.886   -1.569   -1.621   -0.521
2025-08-31 08:02:31.884 2025-09-05 12:00:00            5 -5785.620 26694.954 -32005.802   -1.599   -1.471   -0.063
2025-08-31 08:02:31.884 2025-09-06 12:00:00            6 -5836.565 26351.209 -32281.558   -1.306   -0.996    0.587
2025-08-31 08:02:31.884 2025-09-07 12:00:00            7 -5898.864 26009.214 -32

In [119]:
for a in [0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0]:
    e = np.linalg.norm(targets_km - a * preds_km, axis=1)
    print(a, e.mean().round(3), np.median(e).round(3), np.percentile(e, 90).round(3))

0 3.449 2.23 8.302
0.25 3.081 1.976 7.421
0.5 2.829 1.775 6.494
0.75 2.726 1.859 6.055
1.0 2.794 1.942 5.752
1.5 3.393 2.574 6.771
2.0 4.394 3.443 8.469


In [120]:
use = df_test['days_offset'].values >= 6
e_hyb = np.where(use, err3d_model, err3d_raw_sgp4)

d = df_test.assign(sgp4=err3d_raw_sgp4, hyb=e_hyb)
groups = {t: (g['sgp4'].values, g['hyb'].values) for t, g in d.groupby('tle_epoch')}
keys = list(groups)
rng = np.random.default_rng(0)

res = []
for _ in range(2000):
    pick = rng.choice(len(keys), len(keys))                  # losujemy całe TLE
    a = np.concatenate([groups[keys[i]][0] for i in pick])
    b = np.concatenate([groups[keys[i]][1] for i in pick])
    res.append((a.mean() - b.mean(), np.median(a) - np.median(b),
                np.percentile(a, 90) - np.percentile(b, 90)))
print(np.percentile(res, [2.5, 50, 97.5], axis=0).round(3))   # kolumny: średnia, mediana, p90; >0 = hybryda lepsza

[[0.375 0.085 1.126]
 [0.733 0.319 2.601]
 [1.109 0.607 3.931]]
